In [55]:
import pandas as pd

In [56]:
df = pd.read_csv('../Data/v2_final.tsv',sep='\t', index=False)
print(df.columns)

Index(['index', 'city', 'country', 'description', 'location', 'state',
       'state_abbrev', 'longitude', 'latitude', 'city_longitude',
       'city_latitude', 'Audio Evidence', 'Audio Reasoning', 'Witness Count',
       'Witness Reasoning', 'Event', 'apparition_types_str',
       'apparition_adj_str', 'unique_apparition_mentions', 'Murder per capita',
       'Violent Crime per capita', 'Property Crime per capita',
       'Undergrad_Grad_Rate', 'HS_Grad_Rate', 'STEM_Grad_Percentage',
       'Visual Evidence', 'Visual Reasoning', 'death_rate_Alzheimer's disease',
       'death_rate_Cancer', 'death_rate_Heart disease',
       'death_rate_Unintentional injuries', 'death_rate_All causes',
       'death_rate_Influenza and pneumonia', 'death_rate_Suicide',
       'death_rate_Kidney disease', 'death_rate_CLRD', 'death_rate_Diabetes',
       'Haunted Places Date', 'Alcohol Deaths', 'Alcohol Deaths Under 21',
       'State', 'time_of_day', 'Daylight Data USNO Navy',
       'Daylight Data Timea

/var/folders/0g/wrn8fh3s6x93w1fnwvvh6x4c0000gn/T/ipykernel_79692/1172722253.py:1: DtypeWarning: Columns (63) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../Data/v2_final.tsv',sep='\t')


In [57]:
df['unique_apparition_mentions'] = df['apparition_types_str'].apply(
    lambda x: sum(1 for t in x.split() if t != "Unknown")
)
print(df[['apparition_types_str','unique_apparition_mentions']])

             apparition_types_str  unique_apparition_mentions
0              ghost figure witch                           3
1                         Unknown                           0
2                          spirit                           1
3                         Unknown                           0
4                          entity                           1
...                           ...                         ...
10975                     Unknown                           0
10976                  ghost mist                           2
10977  apparition poltergeist orb                           3
10978                       ghost                           1
10979             figure presence                           2

[10980 rows x 2 columns]


In [60]:
df.to_csv('../Data/v2_final.tsv', sep='\t', index=False)
print("'../Data/v2_final.tsv' downloaded")

'../Data/v2_final.tsv' downloaded


In [70]:
# Step 1: Clean apparition types and add a column per row
df['cleaned_apparitions'] = df['apparition_types_str'].apply(
    lambda x: [t for t in x.split() if t != "Unknown"] if isinstance(x, str) else []
)

# Function to flatten and combine apparition lists per group
def flatten_apparitions(apparition_series):
    return ", ".join(sum(apparition_series, []))

# Function to get unique apparition types per group
def unique_apparition_types(apparition_lists):
    return ", ".join(sorted(set(sum(apparition_lists, []))))

# Function to count the occurrence of each apparition type
def count_apparition_types(apparition_lists):
    all_types = sum(apparition_lists, [])
    return {t: all_types.count(t) for t in set(all_types)}

# Grouping by city, state, and state_abbrev
city_counts = df.groupby(['city', 'state', 'state_abbrev', 'city_latitude', 'city_longitude']).agg(
    # Flatten apparition lists into a single string
    combined_apparitions=('cleaned_apparitions', flatten_apparitions),
    # Sum the unique apparition mentions (total count)
    apparition_mentions=('unique_apparition_mentions', 'sum'),
    # Count the number of rows (haunted places)
    haunted_place_count=('cleaned_apparitions', 'count'),
    # Get the unique apparition types per group
    unique_apparition_types=('cleaned_apparitions', unique_apparition_types),
    # Count occurrences of each apparition type
    apparition_type_counts=('cleaned_apparitions', count_apparition_types)
).reset_index()

# Add the apparition to place ratio
city_counts['apparition:places'] = city_counts['apparition_mentions'] / city_counts['haunted_place_count']

# Preview the result
print(city_counts[['city', 'state', 'state_abbrev', 'city_latitude','city_longitude','haunted_place_count', 
                   'combined_apparitions','apparition_mentions', 'unique_apparition_types', 'apparition_type_counts',
                    'apparition:places']].head())

          city           state state_abbrev  city_latitude  city_longitude  \
0       ARNOLD        Missouri           MO      38.432832      -90.377619   
1    Abbeville  South Carolina           SC      34.178172      -82.379015   
2  Abercrombie    North Dakota           ND      46.447738      -96.730356   
3     Aberdeen        Maryland           MD      39.509556      -76.164120   
4     Aberdeen  North Carolina           NC      35.131547      -79.429479   

   haunted_place_count      combined_apparitions  apparition_mentions  \
0                    1  shadow, figure, presence                    3   
1                    1                     ghost                    1   
2                    1                                              0   
3                    1                    figure                    1   
4                    1                                              0   

    unique_apparition_types                     apparition_type_counts  \
0  figure, presenc

In [71]:
city_counts.sort_values('apparition_mentions', ascending=False).head(10)

,city,state,state_abbrev,city_latitude,city_longitude,combined_apparitions,apparition_mentions,haunted_place_count,unique_apparition_types,apparition_type_counts,apparition:places
2795,Los Angeles,California,CA,34.052234,-118.243685,"figure, spirit, ghost, figure, apparition, gho...",43,61,"apparition, entity, figure, ghost, phantom, pr...","{'figure': 6, 'shadow': 2, 'entity': 1, 'appar...",0.704918
3847,Pittsburgh,Pennsylvania,PA,40.440625,-79.995886,"apparition, mist, ghost, spirit, apparition, s...",30,42,"apparition, figure, ghost, mist, phantom, polt...","{'figure': 2, 'specter': 1, 'mist': 3, 'shadow...",0.714286
4245,San Antonio,Texas,TX,29.424122,-98.493628,"ghost, shadow, ghost, shadow, ghost, spirit, g...",26,55,"apparition, ghost, presence, shadow, spirit","{'shadow': 3, 'apparition': 4, 'presence': 2, ...",0.472727
3622,Orlando,Florida,FL,28.538335,-81.379237,"apparition, shadow, figure, figure, spirit, gh...",26,32,"apparition, figure, ghost, orb, poltergeist, p...","{'figure': 5, 'shadow': 2, 'orb': 1, 'polterge...",0.812500
1458,El Paso,Texas,TX,31.761878,-106.485022,"ghost, figure, presence, figure, mist, mist, g...",22,37,"apparition, figure, ghost, mist, presence, sha...","{'figure': 4, 'mist': 2, 'shadow': 4, 'apparit...",0.594595
3984,Providence,Rhode Island,RI,41.823989,-71.412834,"spirit, ghost, figure, ghost, spirit, appariti...",22,14,"apparition, entity, figure, ghost, orb, polter...","{'figure': 1, 'shadow': 2, 'poltergeist': 2, '...",1.571429
895,Chicago,Illinois,IL,41.878114,-87.629798,"ghost, phantom, ghost, ghost, spirit, ghost, a...",21,30,"apparition, figure, ghost, mist, orb, phantom,...","{'figure': 1, 'shadow': 3, 'orb': 1, 'mist': 1...",0.700000
2223,Hollywood,California,CA,34.092809,-118.328661,"spirit, ghost, ghost, ghost, presence, ghost, ...",21,22,"apparition, entity, figure, ghost, presence, s...","{'figure': 2, 'shadow': 2, 'entity': 2, 'appar...",0.954545
2259,Houston,Texas,TX,29.760427,-95.369803,"presence, apparition, figure, spirit, shadow, ...",21,33,"apparition, figure, ghost, mist, orb, presence...","{'figure': 3, 'shadow': 2, 'mist': 2, 'orb': 1...",0.636364
4253,San Francisco,California,CA,37.774929,-122.419415,"spirit, figure, ghost, spirit, ghost, ghost, g...",19,27,"apparition, figure, ghost, presence, shadow, s...","{'figure': 5, 'shadow': 1, 'apparition': 1, 'p...",0.703704


In [72]:
city_counts.describe()

,city_latitude,city_longitude,apparition_mentions,haunted_place_count,apparition:places
count,5427.000000,5427.000000,5427.000000,5427.000000,5427.000000
mean,38.785566,-90.302669,1.185554,2.017874,0.578217
std,4.858808,15.384874,2.099978,2.913334,0.662159
min,19.575619,-164.723889,0.000000,1.000000,0.000000
25%,35.450528,-96.566247,0.000000,1.000000,0.000000
50%,39.713675,-86.118306,1.000000,1.000000,0.500000
75%,42.059575,-79.971210,1.000000,2.000000,1.000000
max,66.898333,-67.840232,43.000000,61.000000,6.000000


In [73]:
city_counts.to_json('../Source/react-ui/public/ApparitionTypes/apparitions_by_city.json', orient='records', indent=2)
print("'../public/ApparitionTypes/apparitions_by_city.json' downloaded")

'../public/ApparitionTypes/apparitions_by_city.json' downloaded


# Apparition Types by State
The apparition types will now be aggregated by states for a higher level look at the data.

In [77]:
# Function to flatten and combine apparition lists per group
def flatten_apparitions(apparition_series):
    return ", ".join(sum(apparition_series, []))

# Function to get unique apparition types per group
def unique_apparition_types(apparition_lists):
    return ", ".join(sorted(set(sum(apparition_lists, []))))

# Function to count the occurrence of each apparition type
def count_apparition_types(apparition_lists):
    all_types = sum(apparition_lists, [])
    return {t: all_types.count(t) for t in set(all_types)}

# Grouping by city, state, and state_abbrev
state_counts = df.groupby(['state', 'state_abbrev']).agg(
    # Flatten apparition lists into a single string
    combined_apparitions=('cleaned_apparitions', flatten_apparitions),
    # Sum the unique apparition mentions (total count)
    apparition_mentions=('unique_apparition_mentions', 'sum'),
    # Count the number of rows (haunted places)
    haunted_place_count=('cleaned_apparitions', 'count'),
    # Get the unique apparition types per group
    unique_apparition_types=('cleaned_apparitions', unique_apparition_types),
    # Count occurrences of each apparition type
    apparition_type_counts=('cleaned_apparitions', count_apparition_types)
).reset_index()

# Add the apparition to place ratio
state_counts['apparition:places'] = city_counts['apparition_mentions'] / city_counts['haunted_place_count']

# Preview the result
print(state_counts[['state', 'state_abbrev','haunted_place_count', 
                   'combined_apparitions','apparition_mentions', 'unique_apparition_types', 'apparition_type_counts',
                    'apparition:places']].head())

        state state_abbrev  haunted_place_count  \
0     Alabama           AL                  224   
1      Alaska           AK                   32   
2     Arizona           AZ                  156   
3    Arkansas           AR                  119   
4  California           CA                 1071   

                                combined_apparitions  apparition_mentions  \
0  ghost, spirit, shadow, figure, shadow, ghost, ...                  120   
1  ghost, ghost, ghost, spirit, spirit, ghost, po...                   17   
2  ghost, shadow, spirit, presence, spirit, appar...                  101   
3  ghost, orb, shadow, apparition, poltergeist, a...                   69   
4  apparition, ghost, ghost, ghost, apparition, g...                  686   

                             unique_apparition_types  \
0  apparition, entity, figure, ghost, imp, mist, ...   
1     apparition, figure, ghost, poltergeist, spirit   
2  apparition, demon, figure, ghost, mist, orb, p...   
3  app

In [79]:
state_counts.describe()

,apparition_mentions,haunted_place_count,apparition:places
count,50.0000,50.000000,50.000000
mean,129.0200,219.600000,0.801833
std,122.4149,198.742989,0.861946
min,17.0000,32.000000,0.000000
25%,51.0000,76.500000,0.000000
50%,93.0000,167.500000,0.937500
75%,154.5000,285.250000,1.000000
max,686.0000,1071.000000,4.000000


In [80]:
state_counts.to_json('../Source/react-ui/public/ApparitionTypes/apparitions_by_states.json', orient='records', indent=2)
print("'../public/ApparitionTypes/apparitions_by_states.json' downloaded")

'../public/ApparitionTypes/apparitions_by_states.json' downloaded


# JSON list of Apparitions

In [87]:
df_apparition = pd.read_csv('../Data/v2_final.tsv', sep='\t')
df_apparition.head()

/var/folders/0g/wrn8fh3s6x93w1fnwvvh6x4c0000gn/T/ipykernel_79692/860204534.py:1: DtypeWarning: Columns (63) have mixed types. Specify dtype option on import or set low_memory=False.
  df_apparition = pd.read_csv('../Data/v2_final.tsv', sep='\t')


,index,city,country,description,location,state,state_abbrev,longitude,latitude,city_longitude,...,Optional_LATITUDE3,Optional_LATITUDE4,Optional_LONGITUDE1,Optional_LONGITUDE2,Optional_LONGITUDE3,Optional_LONGITUDE4,Optional_NAME1,Optional_NAME2,Optional_NAME3,Optional_NAME4
0,0,Ada,United States,Ada witch - Sometimes you can see a misty blue...,Ada Cemetery,Michigan,MI,-85.504893,42.962106,-85.495480,...,NaN,NaN,-85.50056,-85.49169,NaN,NaN,Egypt Valley Country Club,Findlay Cemetery,NaN,NaN
1,1,Addison,United States,A little girl was killed suddenly while waitin...,North Adams Rd.,Michigan,MI,-84.381843,41.971425,-84.347168,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Adrian,United States,If you take Gorman Rd. west towards Sand Creek...,Ghost Trestle,Michigan,MI,-84.035656,41.904538,-84.037166,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Adrian,United States,"In the 1970's, one room, room 211, in the old ...",Siena Heights University,Michigan,MI,-84.017565,41.905712,-84.037166,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,Albion,United States,Kappa Delta Sorority - The Kappa Delta Sororit...,Albion College,Michigan,MI,-84.745177,42.244006,-84.753030,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [90]:
df_apparition_filtered = df_apparition[['city', 'state', 'city_longitude',
    'city_latitude', 'Audio Evidence', 'Witness Count',
    'Event', 'apparition_types_str',
    'apparition_adj_str', 'unique_apparition_mentions',
    'Visual Evidence', 'time_of_day']]

In [91]:
df_apparition_filtered.to_json('../Source/react-ui/public/ApparitionTypes/apparition_features.json', orient='records', indent=2)
print("'../public/ApparitionTypes/apparition_features.json' downloaded")

'../public/ApparitionTypes/apparition_features.json' downloaded
